## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [13]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Google Generative AI encoder
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [14]:
load_dotenv(override=True)

True

In [15]:
# price is a factor for our company, so we're going to use a low cost model
# grok-4-1-fast-reasoning
MODEL = "grok-4-1-fast-reasoning"
db_name = "vector_db"

grok_api_key = os.getenv('GROK_API_KEY')
grok_base_url = os.getenv('GROK_BASE_URL')

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:8]}")
else:
    print("Grok API Key not set")

print(f"Grok Base URL: {grok_base_url}")


Grok API Key exists and begins xai-Fyau
Grok Base URL: https://api.x.ai/v1


In [16]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [17]:
# How many tokens in all the documents?

try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    print("Model not found in tiktoken, falling back to cl100k_base")
    encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Model not found in tiktoken, falling back to cl100k_base
Total tokens for grok-4-1-fast-reasoning: 63,721


In [18]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [19]:
documents[1]

Document(metadata={'source': 'knowledge-base/company/culture.md', 'doc_type': 'company'}, page_content="# Insurellm Culture\n\n## Vision Statement\nTo revolutionize the insurance industry through innovative technology that makes insurance accessible, transparent, and effortless for everyone.\n\n## Mission Statement\nWe empower insurance providers and consumers with cutting-edge software solutions that streamline processes, enhance customer experiences, and drive meaningful connections in the insurance marketplace. By combining deep industry expertise with technological innovation, we're building the future of insurance.\n\n## Core Values\n\n### Innovation First\nWe challenge the status quo and embrace creative problem-solving. Our team is encouraged to experiment, take calculated risks, and push the boundaries of what's possible in insurance technology. We believe that breakthrough solutions come from curiosity, collaboration, and a willingness to learn from both successes and failures

In [20]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 423 chunks
First chunk:

page_content='# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.

After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.

### Our Culture' metadata={'source': 'knowledge-base/company/careers.md', 'doc_type': 'company'}


In [21]:
chunks[100]

Document(metadata={'source': 'knowledge-base/contracts/Contract with GlobalRe Partners for Rellm.md', 'doc_type': 'contracts'}, page_content="---\n\n## Features\n\nGlobalRe Partners will receive the complete Rellm Enterprise suite with advanced reinsurance capabilities:\n\n1. **Unlimited Reinsurance Administration:** Full support for GlobalRe's 450+ treaty relationships and 8,000+ annual facultative placements with scalability to 1,000+ treaties and 50,000+ facultative certificates.\n\n2. **White-Label Platform:** Complete branding customization:\n   - Custom domains (portal.globalrepartners.com, cedents.globalrepartners.com, brokers.globalrepartners.com)\n   - Branded portals for cedents, brokers, and retrocessionaires\n   - Multi-language interface (English, Spanish, French, German, Mandarin, Japanese, Portuguese, Dutch, Italian, Korean, Arabic, Russian)\n   - Customized reporting templates and market presentations")

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [22]:
import time
from langchain_community.vectorstores import Chroma

## FOR GEMINI - not working from free tier though
gemini_key = os.getenv("GEMINI_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=gemini_key)

print("gemini_key: ", gemini_key[:4])

# 1. Initialize an empty vectorstore
vectorstore = Chroma(
    embedding_function=embeddings, 
    persist_directory=db_name
)

# 2. Define a batch size smaller than the limit (e.g., 10 documents)
batch_size = 15 

# 3. Loop through your chunks and add them with a delay
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    print(f"Adding batch {i//batch_size + 1} ({len(batch)} documents)...")
    
    vectorstore.add_documents(batch)
    
    # Sleep for 10 seconds to respect the 15 RPM limit 
    # (adjust sleep time if you still hit errors)
    time.sleep(10)

print(f"Vectorstore finished with {vectorstore._collection.count()} documents")

gemini_key:  AIza
Adding batch 1 (15 documents)...
Adding batch 2 (15 documents)...
Adding batch 3 (15 documents)...
Adding batch 4 (15 documents)...
Adding batch 5 (15 documents)...
Adding batch 6 (15 documents)...
Adding batch 7 (15 documents)...
Adding batch 8 (15 documents)...
Adding batch 9 (15 documents)...
Adding batch 10 (15 documents)...
Adding batch 11 (15 documents)...
Adding batch 12 (15 documents)...
Adding batch 13 (15 documents)...
Adding batch 14 (15 documents)...
Adding batch 15 (15 documents)...
Adding batch 16 (15 documents)...
Adding batch 17 (15 documents)...
Adding batch 18 (15 documents)...
Adding batch 19 (15 documents)...
Adding batch 20 (15 documents)...
Adding batch 21 (15 documents)...
Adding batch 22 (15 documents)...
Adding batch 23 (15 documents)...
Adding batch 24 (15 documents)...
Adding batch 25 (15 documents)...
Adding batch 26 (15 documents)...
Adding batch 27 (15 documents)...
Adding batch 28 (15 documents)...
Adding batch 29 (3 documents)...
Vector

In [23]:
# Pick an embedding model

#embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# if os.path.exists(db_name):
#     Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
# print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [24]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 423 vectors with 3,072 dimensions in the vector store


### Part C: Visualize!

In [25]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [30]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

# --- ADD THESE TWO LINES TO FIX THE RENDERER ERROR ---
import plotly.io as pio
pio.renderers.default = "notebook_connected"
# -----------------------------------------------------

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [29]:
# Let's try 3D!
# --- ADD THESE TWO LINES TO FIX THE RENDERER ERROR ---
import plotly.io as pio
pio.renderers.default = "notebook_connected"
# -----------------------------------------------------

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()